# ParisiJax Quickstart: From Hamiltonian to Phase Transition

This notebook walks through the core workflow:
1. Generate an SK spin glass instance
2. Scan the RS free energy across temperatures
3. Find the critical temperature
4. Run MCMC at two temperatures (above and below T_c)
5. Plot the overlap distribution P(q)

**Runtime:** < 2 min on CPU

In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

from parisijax.analysis.overlap import compute_overlap
from parisijax.core.hamiltonian import sample_couplings
from parisijax.core.mcmc import run_mcmc
from parisijax.core.solver import find_critical_temperature, high_temp_free_energy, rs_free_energy

print(f"JAX version: {jax.__version__}")
print(f"Devices: {jax.devices()}")

## 1. Generate an SK Instance

In [ ]:
key = jax.random.PRNGKey(42)
N = 128  # System size
J = sample_couplings(key, N, 1)[0]
print(f"Coupling matrix shape: {J.shape}")
print(f"J is symmetric: {jnp.allclose(J, J.T)}")
print(f"J diagonal is zero: {jnp.allclose(jnp.diag(J), 0.0)}")

## 2. RS Free Energy vs Temperature

In [ ]:
betas = jnp.linspace(0.2, 3.0, 30)
f_rs = [float(rs_free_energy(b)) for b in betas]
f_ht = [float(high_temp_free_energy(b)) for b in betas]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(np.array(betas), f_rs, 'o-', label='RS free energy', markersize=4)
ax.plot(np.array(betas), f_ht, '--', label=r'High-$T$: $-\log 2/\beta - \beta/4$', alpha=0.7)
ax.axvline(1.0, color='red', linestyle=':', alpha=0.5, label=r'$\beta_c = 1$')
ax.set_xlabel(r'Inverse temperature $\beta$')
ax.set_ylabel('Free energy per spin')
ax.set_title('SK Model: RS Free Energy')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Find the Critical Temperature

In [ ]:
beta_c = float(find_critical_temperature())
print(f"Critical temperature: beta_c = {beta_c:.4f} (theory: 1.0)")
print(f"T_c = {1/beta_c:.4f}")

## 4. MCMC at Two Temperatures

In [ ]:
n_samples = 200
n_steps = 3000

# High temperature (paramagnetic phase)
key1, key2 = jax.random.split(jax.random.PRNGKey(0))
spins_hot, energies_hot = run_mcmc(key1, J, beta=0.5, n_samples=n_samples, n_steps=n_steps)

# Low temperature (spin glass phase)
spins_cold, energies_cold = run_mcmc(key2, J, beta=2.0, n_samples=n_samples, n_steps=n_steps)

print(f"High-T final energy: {float(jnp.mean(energies_hot[:, -1])):.3f}")
print(f"Low-T final energy:  {float(jnp.mean(energies_cold[:, -1])):.3f}")

## 5. Overlap Distribution P(q)

In [ ]:
# Compute pairwise overlaps
q_hot = jax.vmap(compute_overlap)(spins_hot[::2], spins_hot[1::2])
q_cold = jax.vmap(compute_overlap)(spins_cold[::2], spins_cold[1::2])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.hist(np.array(q_hot), bins=40, range=(-1, 1), density=True, alpha=0.7, edgecolor='black')
ax1.set_title(r'$P(q)$ at $\beta = 0.5$ (paramagnetic)')
ax1.set_xlabel('Overlap $q$')
ax1.set_ylabel('$P(q)$')
ax1.grid(True, alpha=0.3)

ax2.hist(np.array(q_cold), bins=40, range=(-1, 1), density=True, alpha=0.7, edgecolor='black', color='C1')
ax2.set_title(r'$P(q)$ at $\beta = 2.0$ (spin glass)')
ax2.set_xlabel('Overlap $q$')
ax2.set_ylabel('$P(q)$')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"High-T <q^2>: {float(jnp.mean(q_hot**2)):.4f} (expect ~1/N = {1/N:.4f})")
print(f"Low-T  <q^2>: {float(jnp.mean(q_cold**2)):.4f} (expect >> 1/N)")